In [ ]:
!pip install langchain langchain-community langchain-google-genai
!pip install chromadb pypdf pandas streamlit python-dotenv
!pip install sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
!pip install langchain-text-splitters
!pip install langchain-evaluation


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement langchain-evaluation (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchain-evaluation


In [35]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.vectorstores import Chroma

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

from pydantic import BaseModel, Field

In [36]:
import os
import tempfile
import streamlit as st  
import pandas as pd
from dotenv import load_dotenv

In [37]:
load_dotenv()

True

In [38]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

## Defining the LLM

In [39]:
pip install google-generativeai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# Quick test to check if the API key is working and to list available models
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [41]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# quick test to check if the llm is working
llm.invoke("Tell me a short joke about AI")

AIMessage(content='Why did the AI break up with the calculator?\n\nBecause it said, "I need someone who can *process* my feelings, not just *calculate* them!"', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf7fd-3f52-7c70-89b0-a413af667a43-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 1618, 'total_tokens': 1626, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1584}})

## Processing the PDF

In [42]:
loader = PyPDFLoader("data\How-is-It-Possible-to-Create-a-New-Idea.pdf")
pages = loader.load()
pages
    

<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
C:\Users\amrut\AppData\Local\Temp\ipykernel_20560\3259013528.py:1: SyntaxWarning: invalid escape sequence '\H'
  loader = PyPDFLoader("data\How-is-It-Possible-to-Create-a-New-Idea.pdf")


[Document(metadata={'producer': 'Mac OS X 10.2.8 Quartz PDFContext', 'creator': 'Word', 'creationdate': '2008-01-25T23:10:31+00:00', 'subject': 'Creative Intelligent Systems: Papers from the AAAI Spring Symposium', 'author': 'Stellan Ohlsson', 'keywords': 'Technical Report SS-08-03', 'moddate': '2008-03-09T13:22:39-07:00', 'title': 'How Is It Possible to Create a New Idea?', 'source': 'data\\How-is-It-Possible-to-Create-a-New-Idea.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='How is It Possible to Create a New Idea? Stellan Ohlsson  University of Illinois at Chicago Department of Psychology, Behavioral Sciences Building 1007 West Harrison Street, Chicago, IL 60607-7137 stellan@uic,edu    Abstract Creativity requires the generation of novel ideas, a process often called insight. A model of this process is proposed. At the macro-level, the model is a standard symbolic architecture for problem solving. Its problem perception component consists of a layered network o

### Splitting the document into smaller chunks

In [43]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,
                                            chunk_overlap=200,
                                            length_function=len,
                                            separators=["\n\n", "\n", " "])
chunks = text_splitter.split_documents(pages)

### Creating Embeddings

In [44]:
def get_embedding_function():
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )
    return embeddings

embedding_function = get_embedding_function()
test_vector = embedding_function.embed_query("cat")

print(len(test_vector))

3072


In [45]:
!python -m pip install langchain


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
# Quick test to check if the embedding function is working
from sklearn.metrics.pairwise import cosine_similarity

vec1 = embedding_function.embed_query("Amsterdam")  # try again with Paris and check similarity
vec2 = embedding_function.embed_query("Coffeeshop")

similarity = cosine_similarity([vec1], [vec2])[0][0]

print(similarity)

0.6077047161341902


### Create Vector Database

In [47]:
import uuid

def create_vectorstore(chunks, embedding_function, vectorstore_path):

    # Create a list of unique ids for each document based on the content
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    
    # Ensure that only unique docs with unique ids are kept
    unique_ids = set()
    unique_chunks = []
    
    unique_chunks = [] 
    for chunk, id in zip(chunks, ids):     
        if id not in unique_ids:       
            unique_ids.add(id)
            unique_chunks.append(chunk) 

    # Create a new Chroma database from the documents
    vectorstore = Chroma.from_documents(documents=unique_chunks, 
                                        ids=list(unique_ids),
                                        embedding=embedding_function, 
                                        persist_directory = vectorstore_path)

    vectorstore.persist()
    
    return vectorstore

In [48]:
# Create vectorstore
vectorstore = create_vectorstore(chunks=chunks, 
                                 embedding_function=embedding_function, 
                                 vectorstore_path="vectorstore_chroma")

## Querying for Relevant Data

In [49]:
# Load vectorstore
vectorstore = Chroma(persist_directory="vectorstore_chroma", embedding_function=embedding_function)

In [50]:
# Create retriever and get relevant chunks
retriever = vectorstore.as_retriever(search_type="similarity")
relevant_chunks = retriever.invoke("What is the title of the paper?")
relevant_chunks

[Document(metadata={'page': 0, 'page_label': '1', 'source': 'data\\How-is-It-Possible-to-Create-a-New-Idea.pdf', 'title': 'How Is It Possible to Create a New Idea?', 'creationdate': '2008-01-25T23:10:31+00:00', 'author': 'Stellan Ohlsson', 'keywords': 'Technical Report SS-08-03', 'total_pages': 6, 'creator': 'Word', 'moddate': '2008-03-09T13:22:39-07:00', 'subject': 'Creative Intelligent Systems: Papers from the AAAI Spring Symposium', 'producer': 'Mac OS X 10.2.8 Quartz PDFContext'}, page_content='How is It Possible to Create a New Idea? Stellan Ohlsson  University of Illinois at Chicago Department of Psychology, Behavioral Sciences Building 1007 West Harrison Street, Chicago, IL 60607-7137 stellan@uic,edu    Abstract Creativity requires the generation of novel ideas, a process often called insight. A model of this process is proposed. At the macro-level, the model is a standard symbolic architecture for problem solving. Its problem perception component consists of a layered network o

In [51]:
# Prompt template
PROMPT_TEMPLATE = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, say that you
don't know. DON'T MAKE UP ANYTHING.

context: {context}

---

Answer the question based on the above context: {question}
"""

In [52]:
# Concatenate context text
context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, 
                                question="What is the title of the paper?")
print(prompt)

Human: 
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, say that you
don't know. DON'T MAKE UP ANYTHING.

context: How is It Possible to Create a New Idea? Stellan Ohlsson  University of Illinois at Chicago Department of Psychology, Behavioral Sciences Building 1007 West Harrison Street, Chicago, IL 60607-7137 stellan@uic,edu    Abstract Creativity requires the generation of novel ideas, a process often called insight. A model of this process is proposed. At the macro-level, the model is a standard symbolic architecture for problem solving. Its problem perception component consists of a layered network of processing units. At the micro-level, each such unit propagates its computational result along a subset of its forward links. The subset is selected by matching the activation levels of the links to a threshold for forward propagation. The activation levels are subject to feedback ad

## Generating Responses

In [53]:
llm.invoke(prompt)

AIMessage(content='The title of the paper is "How is It Possible to Create a New Idea?".', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf7fd-fd1d-7190-bf73-f7e39ede134e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 889, 'output_tokens': 50, 'total_tokens': 939, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 33}})

### Using LangChain Expression Language


In [54]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt_template
            | llm
        )
rag_chain.invoke("What's the title of this paper?")

AIMessage(content='The title of this paper is "How is It Possible to Create a New Idea?".', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf7fe-09ee-7761-9941-fa984701b073-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 881, 'output_tokens': 51, 'total_tokens': 932, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 34}})

### Generating Structured Responses

In [55]:
class AnswerWithSources(BaseModel):
    """An answer to the question, with sources and reasoning."""
    answer: str = Field(description="Answer to question")
    sources: str = Field(description="Full direct text chunk from the context used to answer the question")
    reasoning: str = Field(description="Explain the reasoning of the answer based on the sources")
    
class ExtractedInfo(BaseModel):
    """Extracted information about the research article"""
    paper_title: AnswerWithSources
    paper_summary: AnswerWithSources
    publication_year: AnswerWithSources
    paper_authors: AnswerWithSources

In [56]:
rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt_template
            | llm.with_structured_output(ExtractedInfo, strict=True)
        )

rag_chain.invoke("Give me the title, summary, publication date, authors of the research paper.")

ExtractedInfo(paper_title=AnswerWithSources(answer='How is It Possible to Create a New Idea?', sources='How is It Possible to Create a New Idea? Stellan Ohlsson University of Illinois at Chicago Department of Psychology, Behavioral Sciences Building 1007 West Harrison Street, Chicago, IL 60607-7137 stellan@uic,edu', reasoning='The title is directly stated at the beginning of the provided text.'), paper_summary=AnswerWithSources(answer="The paper proposes a model for the generation of novel ideas, a process often called insight, which is a key component of creativity. The model uses a standard symbolic architecture for problem solving at the macro-level, with a layered network of processing units. At the micro-level, these units propagate computational results based on matching activation levels to a threshold, which is subject to feedback adjustment and re-distribution. A simulation demonstrates that these characteristics can lead to sudden alterations in a unit's output, suggesting ho

### Structured Response to Dataframe

In [57]:
structured_response = rag_chain.invoke("Give me the title, summary, publication date, authors of the research paper.")
df = pd.DataFrame([structured_response.dict()])

# Transforming into a table with two rows: 'answer' and 'source'
answer_row = []
source_row = []
reasoning_row = []

for col in df.columns:
    answer_row.append(df[col][0]['answer'])
    source_row.append(df[col][0]['sources'])
    reasoning_row.append(df[col][0]['reasoning'])

# Create new dataframe with two rows: 'answer' and 'source'
structured_response_df = pd.DataFrame([answer_row, source_row, reasoning_row], columns=df.columns, index=['answer', 'source', 'reasoning'])
structured_response_df

C:\Users\amrut\AppData\Local\Temp\ipykernel_20560\1998849655.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  df = pd.DataFrame([structured_response.dict()])


,paper_title,paper_summary,publication_year,paper_authors
answer,How is It Possible to Create a New Idea?,The paper proposes a model for the generation ...,2007,Stellan Ohlsson
source,How is It Possible to Create a New Idea? Stell...,Abstract Creativity requires the generation of...,"Copyright © 2007, Association for the Advancem...",How is It Possible to Create a New Idea? Stell...
reasoning,The title is directly stated at the very begin...,The summary is constructed by combining inform...,The publication year is identified from the co...,"The author's name, Stellan Ohlsson, is listed ..."
